In [8]:
import numpy as np
import torch
import torch.nn as nn
from deep_hedging_env import HedgingEnv
from logit_normal import LogitNormal
from reward_utils import compute_discounted_cumsum_rewards
from plot_utils import plot_portfolio_vs_option_price

### Policy Gradient: Simple MLP

In [9]:
class PolicyNetwork(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_size,
        action_dim=1,
    ):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        self.fc_sigma = nn.Linear(hidden_size, action_dim)
        self.softplus = nn.Softplus()

    def forward(self, history_features):

        x = history_features[
            :, -1, :
        ]  # simple MLP just uses the latest state's feature
        x = torch.tanh(self.fc1(x))
        mu = self.fc_mu(x)
        sigma = self.softplus(self.fc_sigma(x))
        return mu, sigma

    def sample_action(self, mu, sigma, deterministic=False):
        logit_normal = LogitNormal(mu, sigma)

        if deterministic:
            action = torch.sigmoid(mu)
        else:
            action = logit_normal.sample()
        log_prob = logit_normal.log_prob(action)

        return action, log_prob

In [10]:
class InactionNet(nn.Module):
    def __init__(
            self,
            input_dim,
            hidden_size,
            action_dim=1,
    ):
        super(InactionNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc2 = nn.Linear(hidden_size, action_dim)
        self.fc_sigma = nn.Linear(hidden_size, action_dim)  # This should take hidden_size input
        self.softplus = nn.Softplus()
    
    def forward(self, history_features):
        x = history_features[
            :, -1, :
        ]  # simple MLP just uses the latest state's feature
        h = torch.tanh(self.fc1(x))  # hidden layer output
        mu = self.fc2(h)             # mu from hidden layer
        sigma = self.softplus(self.fc_sigma(h))  # sigma from hidden layer

        return mu, sigma

    def sample_action(self, mu, sigma, deterministic=False):
        logit_normal = LogitNormal(mu, sigma)
        if not deterministic:
            action = logit_normal.sample()
        else:
            action = torch.sigmoid(mu)
        
        log_prob = logit_normal.log_prob(action)
        return action, log_prob

In [ ]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
sigma = np.array([0.15, 0.2, 0.25])
r = 0.05
num_simulation = 100
num_step = 250
data_generation = "heston"


env = HedgingEnv(
    S0, K, sigma, r, num_simulation=num_simulation, num_step=num_step, data_generation=data_generation
)

# --- Policy Network Parameters ---
input_dim = 11
hidden_size = 64
history_len = 15

policy_net = PolicyNetwork(input_dim, hidden_size)
inaction_net = InactionNet(input_dim, hidden_size)

# --- Optimization Parameters ---
learning_rate = 1e-3

optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)
optimizer_2 = torch.optim.Adam(inaction_net.parameters(), lr=learning_rate)


# --- Other Parameters ---
num_episodes = 200
num_epochs = 10
discount_factor = 0.999

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)
inaction_net.to(device)

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        log_prob_history = []
        inaction_log_prob_history = []  # ADD: Track inaction decisions
        reward_history = []
        state_history = []

        state, _ = env.reset(seed=epoch + 1000)  # [num_envs, obs_dim]
        state = state[:, None, :]  # [num_envs, 1, obs_dim]
        state_history.append(state)

        done = np.zeros(env.num_envs, dtype=bool)

        while not all(done):
            # Prepare input to policy and inaction net
            policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
            policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)

            # Get action
            action_mu, action_sigma = policy_net(policy_net_input_tensor)
            action, log_prob = policy_net.sample_action(action_mu, action_sigma)
            log_prob_history.append(log_prob)
            action_np = action.detach().cpu().numpy()

            # Decide which envs to step using inaction_net
            inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
            inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma)
            inaction_log_prob_history.append(inaction_log_prob)
            
            # Convert inaction action to boolean decision (> 0.5 means take action)
            should_step = (inaction_action > 0.5).cpu().numpy()  # boolean mask

            # Step all environments
            next_state_all, reward_all, done_all, _, _ = env.step(action_np)

            # Initialize new buffers
            next_state = np.copy(state[:, 0, :])  # [num_envs, obs_dim]
            reward = np.zeros(env.num_envs)
            new_done = np.copy(done)

            # Apply step results only where allowed
            for i in range(env.num_envs):
                if not done[i]:
                    if should_step[i]:
                        next_state[i] = next_state_all[i]
                        reward[i] = reward_all[i]
                        new_done[i] = done_all[i]
                    else:
                        # Keep state, give same reward as if stepped, but still check if episode should end
                        reward[i] = reward_all[i]
                        new_done[i] = done_all[i]

            # Update trackers
            state = next_state[:, None, :]  # [num_envs, 1, obs_dim]
            state_history.append(state)
            reward_history.append(reward)
            done = new_done

        # Compute and normalize rewards
        R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)  # shape: [time, num_envs]
        R = R - R.mean(axis=1, keepdims=True)
        R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
        R = torch.tensor(R, dtype=torch.float32).to(device)

        # Compute loss and update
        optimizer.zero_grad()
        optimizer_2.zero_grad()
        
        # Policy loss
        policy_loss = (-R * torch.stack(log_prob_history)).mean()
        
        # ADD: Inaction loss  
        inaction_loss = (-R * torch.stack(inaction_log_prob_history)).mean()
        
        policy_loss.backward()
        inaction_loss.backward()
        
        optimizer.step()
        optimizer_2.step()

        if (episode + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, "
                f"Episode {episode + 1}/{num_episodes}, "
                f"Policy Loss: {policy_loss.item():.4f}, "
                f"Inaction Loss: {inaction_loss.item():.4f}, "
                f"Avg. Reward: {np.array(reward_history).mean():.4f}"
            )

Epoch 1/10, Episode 10/200, Policy Loss: -0.0046, Inaction Loss: -0.0078, Avg. Reward: -4.1802
Epoch 1/10, Episode 20/200, Policy Loss: -0.0061, Inaction Loss: -0.0064, Avg. Reward: -4.1441
Epoch 1/10, Episode 30/200, Policy Loss: -0.0067, Inaction Loss: -0.0052, Avg. Reward: -4.1725
Epoch 1/10, Episode 40/200, Policy Loss: -0.0090, Inaction Loss: -0.0042, Avg. Reward: -4.0840
Epoch 1/10, Episode 50/200, Policy Loss: -0.0135, Inaction Loss: -0.0061, Avg. Reward: -3.9753
Epoch 1/10, Episode 60/200, Policy Loss: -0.0113, Inaction Loss: -0.0057, Avg. Reward: -3.7796
Epoch 1/10, Episode 70/200, Policy Loss: -0.0094, Inaction Loss: -0.0071, Avg. Reward: -3.7284
Epoch 1/10, Episode 80/200, Policy Loss: -0.0093, Inaction Loss: -0.0027, Avg. Reward: -3.5232
Epoch 1/10, Episode 90/200, Policy Loss: -0.0250, Inaction Loss: -0.0045, Avg. Reward: -3.3869
Epoch 1/10, Episode 100/200, Policy Loss: -0.0305, Inaction Loss: -0.0033, Avg. Reward: -3.3885
Epoch 1/10, Episode 110/200, Policy Loss: -0.0316

In [5]:
# Train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        log_prob_history = []
        reward_history = []
        state_history = []

        state, _ = env.reset(seed=epoch+1000) # avoid using seed=0 as it's for testing
        state = state[:, None, :]
        state_history.append(state)

        while True:
            policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
            policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(
                device
            )
            action_mu, action_sigma = policy_net(policy_net_input)
            action, log_prob = policy_net.sample_action(action_mu, action_sigma)
            log_prob_history.append(log_prob)
            action_np = action.detach().cpu().numpy()
            state, reward, done, _, _ = env.step(action_np)
            reward_history.append(reward)
            state = state[:, None, :]
            state_history.append(state)

            if all(done):
                break

        R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)
        R = R - R.mean(axis=1, keepdims=True)
        R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
        R = torch.tensor(R, dtype=torch.float32).to(device)
        optimizer.zero_grad()
        loss = (-R * torch.stack(log_prob_history)).mean()
        loss.backward()
        optimizer.step()

        if (episode + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss.item()}, Avg. Reward: {np.array(reward_history).mean()}"
            )

Epoch 1/10, Episode 10/200, Loss: 0.009731108322739601, Avg. Reward: -4.166353319505017
Epoch 1/10, Episode 20/200, Loss: 0.011425665579736233, Avg. Reward: -4.0822180511597805
Epoch 1/10, Episode 30/200, Loss: 0.01781194470822811, Avg. Reward: -3.960044526789738
Epoch 1/10, Episode 40/200, Loss: 0.014596761204302311, Avg. Reward: -4.004089077248404
Epoch 1/10, Episode 50/200, Loss: 0.010379429906606674, Avg. Reward: -3.945463317520527
Epoch 1/10, Episode 60/200, Loss: 0.008187379688024521, Avg. Reward: -3.881260315969292
Epoch 1/10, Episode 70/200, Loss: 0.007121242582798004, Avg. Reward: -3.779854333426188
Epoch 1/10, Episode 80/200, Loss: -0.003413780825212598, Avg. Reward: -3.6739210370816964
Epoch 1/10, Episode 90/200, Loss: -0.00898695271462202, Avg. Reward: -3.523188173927144
Epoch 1/10, Episode 100/200, Loss: -0.015781845897436142, Avg. Reward: -3.328067280604808
Epoch 1/10, Episode 110/200, Loss: -0.013666640967130661, Avg. Reward: -3.5163574608977215
Epoch 1/10, Episode 120/2

KeyboardInterrupt: 

In [ ]:
action_sigma.min(), action_sigma.max(), action_sigma.mean(), action.min(), action.max(), action.mean()

(tensor(0.0534, grad_fn=<MinBackward1>),
 tensor(0.3642, grad_fn=<MaxBackward1>),
 tensor(0.1219, grad_fn=<MeanBackward0>),
 tensor(0.0208),
 tensor(0.9836),
 tensor(0.5582))

In [37]:
# Test

env = HedgingEnv(S0, K, sigma, r, num_simulation=5, num_step=num_step)

log_prob_history = []
reward_history = []
state_history = []

state, _ = env.reset(seed=0)
state = state[:, None, :]
state_history.append(state)

while True:
    policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
    policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
    action_mu, action_sigma = policy_net(policy_net_input)
    action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
    log_prob_history.append(log_prob)
    action_np = action.detach().cpu().numpy()
    state, reward, done, _, _ = env.step(action_np)
    reward_history.append(reward)
    state = state[:, None, :]
    state_history.append(state)

    if all(done):
        break

In [35]:
# Test

env = HedgingEnv(S0, K, sigma, r, num_simulation=5, num_step=num_step)

log_prob_history = []
inaction_log_prob_history = []  # ADD: Also track inaction decisions in test
reward_history = []
state_history = []
action_taken_history = []  # Track when actions were actually taken

state, _ = env.reset(seed=0)
state = state[:, None, :]
state_history.append(state)

done = np.zeros(env.num_envs, dtype=bool)

while not all(done):
    policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
    policy_net_input_tensor = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
    
    # Get action from policy network
    action_mu, action_sigma = policy_net(policy_net_input_tensor)
    action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
    log_prob_history.append(log_prob)
    action_np = action.detach().cpu().numpy()
    
    # Get inaction decision - FIX: Properly unpack the tuple like in training
    inaction_mu, inaction_sigma = inaction_net(policy_net_input_tensor)
    inaction_action, inaction_log_prob = inaction_net.sample_action(inaction_mu, inaction_sigma)
    inaction_log_prob_history.append(inaction_log_prob)
    
    # Convert inaction action to boolean decision (> 0.5 means take action)
    should_step = (inaction_action > 0.5).cpu().numpy()  # boolean mask
    action_taken_history.append(should_step.copy())  # Track for later analysis
    
    # Step all environments
    next_state_all, reward_all, done_all, _, _ = env.step(action_np)
    
    # Initialize new buffers
    next_state = np.copy(state[:, 0, :])
    reward = np.zeros(env.num_envs)
    new_done = np.copy(done)
    
    # Apply step results only where allowed
    for i in range(env.num_envs):
        if not done[i]:
            if should_step[i]:
                next_state[i] = next_state_all[i]
                reward[i] = reward_all[i]
                new_done[i] = done_all[i]
            else:
                # Keep state, give same reward as if stepped, but still check if episode should end
                reward[i] = reward_all[i]
                new_done[i] = done_all[i]
    
    # Update trackers
    state = next_state[:, None, :]
    state_history.append(state)
    reward_history.append(reward)
    done = new_done

print(f"Test completed. Total reward: {np.array(reward_history).sum():.4f}")
print(f"Average reward per step: {np.array(reward_history).mean():.4f}")
print(f"Actions taken: {np.mean([step.sum() for step in action_taken_history]):.2f} out of {env.num_envs} environments per step")

Test completed. Total reward: -7423.4835
Average reward per step: -0.9898
Actions taken: 19.38 out of 30 environments per step


In [36]:
rewards = np.array(reward_history)
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(-8.459365890539395,
 -0.0001605089123231096,
 -0.98979780085324,
 1.130160794416858)

In [37]:
plot_portfolio_vs_option_price(env)

## Recurent

In [10]:
class PolicyNetwork(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_size,
        num_layers,
        action_dim=1,
        history_len=5,
        dropout=0.0,
    ):
        super(PolicyNetwork, self).__init__()
        self.history_len = history_len
        self.rnn = nn.GRU(
            input_dim, hidden_size, num_layers, batch_first=True, dropout=dropout
        )
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        self.fc_sigma = nn.Linear(hidden_size, action_dim)
        self.softplus = nn.Softplus()

    def forward(self, history_features, determistic=False):
        # history_features: (batch_size, history_len, feature_dim)

        batch_size = history_features.size(0)
        seq_len = history_features.size(1)

        # Pad history if shorter than history_len
        if seq_len < self.history_len:
            padding = torch.zeros(
                batch_size,
                self.history_len - seq_len,
                history_features.size(2),
                dtype=history_features.dtype,
                device=history_features.device,
            )
            history_features = torch.cat([padding, history_features], dim=1)

        output, _ = self.rnn(
            history_features
        )  # out: tensor of shape (batch_size, seq_length, hidden_size)
        output = output[:, -1, :]  # Take output from the last time step

        mu = self.fc_mu(output)
        sigma = self.softplus(self.fc_sigma(output))

        return mu, sigma

    def sample_action(self, mu, sigma, deterministic=False):
        logit_normal = LogitNormal(mu, sigma)

        if deterministic:
            action = torch.sigmoid(mu)
        else:
            action = logit_normal.sample()
        log_prob = logit_normal.log_prob(action)

        return action, log_prob

In [11]:
# --- Env. Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
sigma = np.array([0.15, 0.2, 0.25])
r = 0.05
num_simulation = 100
num_step = 250
data_generation = "heston"

env = HedgingEnv(
    S0, K, sigma, r, num_simulation=num_simulation, num_step=num_step, data_generation = data_generation
)

# --- Policy Network Parameters ---
input_dim = 11
hidden_size = 64
num_layers = 2
history_len = 15

policy_net = PolicyNetwork(input_dim, hidden_size, num_layers, history_len=history_len)

# --- Optimization Parameters ---
learning_rate = 1e-4

optimizer = torch.optim.Adam(policy_net.parameters(), lr=learning_rate)

# --- Other Parameters ---
num_episodes = 200
num_epochs = 10
discount_factor = 0.999

In [ ]:
# Train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)

for epoch in range(num_epochs):
    for episode in range(num_episodes):
        log_prob_history = []
        reward_history = []
        state_history = []

        state, _ = env.reset(seed=epoch+1000) # avoid using seed=0 as it's for testing
        state = state[:, None, :]
        state_history.append(state)

        while True:
            policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
            policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(
                device
            )
            action_mu, action_sigma = policy_net(policy_net_input)
            action, log_prob = policy_net.sample_action(action_mu, action_sigma)
            log_prob_history.append(log_prob)
            action_np = action.detach().cpu().numpy()
            state, reward, done, _, _ = env.step(action_np)
            reward_history.append(reward)
            state = state[:, None, :]
            state_history.append(state)

            if all(done):
                break

        R = compute_discounted_cumsum_rewards(np.array(reward_history), discount_factor)
        R = R - R.mean(axis=1, keepdims=True)
        R = R / (R.std(axis=1, keepdims=True) + np.finfo(R.dtype).eps)
        R = torch.tensor(R, dtype=torch.float32).to(device)
        optimizer.zero_grad()
        loss = (-R * torch.stack(log_prob_history)).mean()
        loss.backward()
        optimizer.step()

        if (episode + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss.item()}, Avg. Reward: {np.array(reward_history).mean()}"
            )

In [ ]:
action_sigma.min(), action_sigma.max(), action_sigma.mean() 

In [ ]:
# Test

env = HedgingEnv(S0, K, sigma, r, num_simulation=5, num_step=num_step)

log_prob_history = []
reward_history = []
state_history = []

state, _ = env.reset(seed=0)
state = state[:, None, :]
state_history.append(state)

while True:
    policy_net_input = np.concatenate(state_history[-history_len:], axis=1)
    policy_net_input = torch.tensor(policy_net_input, dtype=torch.float32).to(device)
    action_mu, action_sigma = policy_net(policy_net_input)
    action, log_prob = policy_net.sample_action(action_mu, action_sigma, True)
    log_prob_history.append(log_prob)
    action_np = action.detach().cpu().numpy()
    state, reward, done, _, _ = env.step(action_np)
    reward_history.append(reward)
    state = state[:, None, :]
    state_history.append(state)

    if all(done):
        break

In [ ]:
rewards = np.array(reward_history)
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

In [ ]:
plot_portfolio_vs_option_price(env)